# 05 — Analyse des erreurs et des prix extrêmes

Cette étape cherche à comprendre où le Ridge se trompe. Les seuils de prix ci-dessous servent uniquement à décrire la queue de distribution : ils ne déclenchent aucune suppression. Les prix DVF peuvent inclure des lignes de terrain ou de dépendance associées au logement.

In [ ]:
import json
from pathlib import Path

import pandas as pd

METRICS_PATH = Path("ml/evaluation/baseline_metrics.json")
DATA_PATH = Path("data/processed/dvf_single_home_candidates_2025.csv")
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Effectifs du test :", metrics["row_counts"])
print("Résultat Ridge :", metrics["metrics"]["ridge_log_target"])

## Où le modèle se trompe-t-il ?

Une erreur moyenne négative signifie que les prédictions sont en dessous des prix observés.

In [ ]:
bands = pd.DataFrame(
    metrics["ridge_metrics_by_actual_price_band"]
).T
display(
    bands[
        [
            "rows",
            "median_actual_price_eur",
            "mae_eur",
            "mean_error_eur",
            "rmse_eur",
        ]
    ].round(0)
)

display(pd.DataFrame(metrics["ridge_metrics_by_property_type"]).T.round(0))
hgb_bands = pd.DataFrame(
    metrics["hgb_metrics_by_actual_price_band"]
).T
print("Histogram Gradient Boosting par tranche :")
display(
    hgb_bands[
        ["rows", "median_actual_price_eur", "mae_eur", "mean_error_eur", "rmse_eur"]
    ].round(0)
)


## Taille et prix au mètre carré

Ces quantiles aident à distinguer les maisons ou appartements simplement coûteux des lignes présentant une combinaison inhabituelle de prix et de surface. Le prix au mètre carré reste lui aussi un indicateur descriptif, pas une vérité terrain indépendante.

In [ ]:
price_quantiles = df.groupby("property_type")["sale_price_eur"].quantile(
    [0.95, 0.99, 0.999, 1.0]
).unstack()
price_quantiles.columns = ["p95", "p99", "p99.9", "max"]
display(price_quantiles.round(0))

price_per_m2 = df["sale_price_eur"] / df["built_area_m2"]
area_price_quantiles = price_per_m2.groupby(df["property_type"]).quantile(
    [0.5, 0.95, 0.99, 0.999]
).unstack()
area_price_quantiles.columns = ["p50", "p95", "p99", "p99.9"]
display(area_price_quantiles.round(0))

threshold_counts = pd.DataFrame(
    {
        f"> {threshold:,.0f} €": df["sale_price_eur"].gt(threshold)
        .groupby(df["property_type"])
        .sum()
        for threshold in [1_000_000, 5_000_000, 10_000_000]
    }
)
display(threshold_counts.astype("int64"))

## Bilan

Le modèle sous-estime surtout le quintile supérieur. Les montants les plus élevés du jeu candidat sont associés à des surfaces bâties très grandes ; l’audit source montre aussi des enregistrements de terrain ou des dépendances dans certains groupes. Cela rend une suppression automatique difficile à justifier.

Un meilleur prochain essai est d’ajouter une caractéristique de surface de terrain, agrégée au niveau du groupe sans compter plusieurs fois la même parcelle. Cette variable doit être auditée avant entraînement : les surfaces peuvent être répétées sur plusieurs lignes DVF et une parcelle peut être partagée entre plusieurs locaux. Gardons les prix extrêmes dans l’évaluation et comparons les performances avec et sans la nouvelle caractéristique.